<div style="background: linear-gradient(135deg, #fdfbfb 0%, #ebedee 100%); border-radius: 10px; border-left: 6px solid #ff9900; box-shadow: 0 3px 6px rgba(0,0,0,0.06); width: 100%; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; overflow: hidden;">
  <table style="border: none; width: 100%; margin: 0; border-collapse: collapse; background: transparent;">
    <tr style="border: none; background: transparent;">
      <td style="border: none; width: 100px; text-align: center; vertical-align: middle; padding: 14px 0 14px 15px; background: transparent;">
        <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
      </td>
      <td style="border: none; vertical-align: middle; padding: 14px 15px; background: transparent;">
        <div style="color: #555555; font-size: 13px; font-weight: 500; margin-bottom: 2px;">Excerpt from the Amazon book</div>
        <div style="font-family: 'Arial Black', Impact, sans-serif; font-size: 18px; font-weight: 900; text-transform: uppercase; letter-spacing: 1.5px; color: #111111; margin: 6px 0 10px 0; line-height: 1.2;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</div>
        <div>
          <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: none; font-weight: 600; font-size: 13px; display: inline-block;">🛒 Buy on Amazon &rarr;</a>
        </div>
      </td>
    </tr>
  </table>
</div>

# **For this notebook to work, you must execute ALL the cells in the order in which they appear.**


# **# 1. Getting files from repository:** This is not necessary when working in Fabric

In [ ]:
!wget -O sales.csv https://raw.githubusercontent.com/sansalamanca/shining-path_DP700/main/data/sales.csv

# Create a Spark session with a custom name.
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName("Colab_Session_5") \
.getOrCreate()


--2026-11-14 14:14:16--  https://raw.githubusercontent.com/sansalamanca/shining-path_DP700/main/data/sales.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3308303 (3.2M) [text/plain]
Saving to: ‘sales.csv’

sales.csv           100%[===================>]   3.15M  --.-KB/s    in 0.07s   

2026-11-14 14:14:16 (45.7 MB/s) - ‘sales.csv’ saved [3308303/3308303]



# **Observe the data frame we are working with**

In [ ]:

from pyspark.sql.types import *

orderSchema = StructType([
   StructField("SalesOrderNumber", StringType()),
   StructField("SalesOrderLineNumber", IntegerType()),
   StructField("OrderDate", DateType()),
   StructField("CustomerName", StringType()),
   StructField("Email", StringType()),
   StructField("Item", StringType()),
   StructField("Quantity", IntegerType()),
   StructField("UnitPrice", FloatType()),
   StructField("Tax", FloatType())
])

df = spark.read.format("csv").schema(orderSchema).load("sales.csv")

df.show()   # In fABRIC USE:    display(df)



+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|     CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|                NULL|      NULL|     CustomerName|        EmailAddress|                Item|    NULL|     NULL|    NULL|
|         SO43701|                   1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|         

# **Query using the Spark DataFrame API (DSL)**

In [ ]:
# Grouping and aggregating using the Spark DataFrame API

from pyspark.sql import functions as F

# We apply the transformations and groupings
resulting_df = df.groupBy("CustomerName") \
  .agg(
      F.countDistinct("SalesOrderNumber").alias("#_Of_Orders"),
      F.sum(F.col("Quantity") * F.col("UnitPrice")).alias("TotalPurchased")
  ) \
  .orderBy(F.col("TotalPurchased").desc())

# Show results
resulting_df.show()   # en Fabric use display(resultado_df)


+------------------+-----------+------------------+
|      CustomerName|#_Of_Orders|    TotalPurchased|
+------------------+-----------+------------------+
|     Larry Vazquez|          4|10899.620180130005|
| Kaitlyn Henderson|          4|10896.220178842545|
|      Nichole Nara|          4|10876.320177316666|
|        Kate Anand|          4| 10872.06017780304|
|       Margaret He|          4|10841.220174789429|
|   Lawrence Alonso|          4|10836.900176048279|
|Terrance Rodriguez|          4|10829.220174789429|
|           Rosa Hu|          4|10817.600177764893|
|      Aaron Wright|          4|10813.630178451538|
|      Clarence Gao|          4|10799.520174980164|
|       Brandi Gill|          4|10797.590178489685|
| Randall Dominguez|          4|10796.940176963806|
|       Bonnie Nath|          4|10793.270175933838|
|  Adriana Gonzalez|          4|10789.660177230835|
|       Andres Nara|          4|10789.530175447464|
|          Brad She|          4|10789.120176315308|
|      Ethan

# **Query using the SQL engine**



In [ ]:
# 1. Register the DataFrame as a temporary view in Spark
df.createOrReplaceTempView("orders_view")

# 2. Write and execute the query using SQL
resulting_df = spark.sql("""
    SELECT
        CustomerName,
        COUNT(DISTINCT SalesOrderNumber) AS _Of_Orders,
        SUM(Quantity * UnitPrice) AS TotalPurchased
    FROM orders_view
    GROUP BY CustomerName
    ORDER BY TotalPurchased DESC
""")

# 3. Display results
resulting_df.show()          # In Fabric use display(resultado_df)

+------------------+----------+------------------+
|      CustomerName|_Of_Orders|    TotalPurchased|
+------------------+----------+------------------+
|     Larry Vazquez|         4|10899.620180130005|
| Kaitlyn Henderson|         4|10896.220178842545|
|      Nichole Nara|         4|10876.320177316666|
|        Kate Anand|         4| 10872.06017780304|
|       Margaret He|         4|10841.220174789429|
|   Lawrence Alonso|         4|10836.900176048279|
|Terrance Rodriguez|         4|10829.220174789429|
|           Rosa Hu|         4|10817.600177764893|
|      Aaron Wright|         4|10813.630178451538|
|      Clarence Gao|         4|10799.520174980164|
|       Brandi Gill|         4|10797.590178489685|
| Randall Dominguez|         4|10796.940176963806|
|       Bonnie Nath|         4|10793.270175933838|
|  Adriana Gonzalez|         4|10789.660177230835|
|       Andres Nara|         4|10789.530175447464|
|          Brad She|         4|10789.120176315308|
|      Ethan Bryant|         4|

# **Group by year and count the number of sales**

In [ ]:
# Group by year and count the number of sales

from pyspark.sql.functions import *
yearlySales = df.select(year(col("OrderDate")).alias("Year")).groupBy("Year").count().orderBy("Year")
yearlySales.show()    # En Fabric use:   display(yearlySales)

+----+-----+
|Year|count|
+----+-----+
|NULL|    1|
|2019| 1201|
|2020| 2733|
|2021|28784|
+----+-----+

